# Step 3. Web Search Agent

확증 편향 방지 쿼리와 검색 제한 로직을 먼저 검증한다.


In [ ]:
from pathlib import Path
import sys
from dotenv import load_dotenv

CURRENT = Path.cwd().resolve()
PROJECT_ROOT = None
for candidate in [CURRENT, *CURRENT.parents]:
    if (candidate / "src" / "semiconductor_agent").exists():
        PROJECT_ROOT = candidate
        break
if PROJECT_ROOT is None:
    raise RuntimeError("project root with src/semiconductor_agent not found")

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

for env_candidate in [PROJECT_ROOT / ".env", PROJECT_ROOT.parent / ".env"]:
    if env_candidate.exists():
        load_dotenv(env_candidate, override=False)
import pandas as pd

from semiconductor_agent.state import build_initial_state
from semiconductor_agent.web_search import build_balanced_queries, register_search_attempt


## 3-1. Balanced Query Set

회사/기술 조합별로 progress/risk를 대칭적으로 수집하는 쿼리를 본다.


In [ ]:
queries = build_balanced_queries("Samsung", "HBM4")
pd.DataFrame({"query": queries})


## 3-2. Search Limit Simulation

web search call limit과 retry limit이 의도대로 동작하는지 실험한다.


In [ ]:
state = build_initial_state("web search simulation")
results = []

for _ in range(5):
    allowed, reason = register_search_attempt(state, "Samsung:HBM4:roadmap")
    results.append(
        {
            "allowed": allowed,
            "reason": reason,
            "retry_count": state["retry_count"].get("search:Samsung:HBM4:roadmap", 0),
            "web_search_call_count": state["web_search_call_count"],
        }
    )

pd.DataFrame(results)
